In [1]:
# External Imports
import sys
from pathlib import Path
import glob
import matplotlib.pyplot as plt
import torch

# Internal Imports
sys.path.insert(0, '../src')
from src.DataIntegrity import data_integrity_check
from src.Dataset import split_patients, MriDataset, get_train_transforms, get_val_transforms, get_dataloaders

## Data Integrity Check

In [2]:
data_path = Path("../data/raw/lgg-mri-segmentation/kaggle_3m")
rejected_path = Path("../data/processed/data-integrity")

In [3]:
print(len(glob.glob("../data/raw/lgg-mri-segmentation/kaggle_3m/*")))

112


In [4]:
accepted_data, rejected_data = data_integrity_check(data_path)

[INFO] Data Integrity Checks:   1%|          | 1/112 [00:20<38:37, 20.88s/it]


KeyboardInterrupt: 

## Split Dataset

In [7]:
SPLIT_SEED = 42

In [8]:
train_patients, val_patients, test_patients = split_patients(accepted_data, seed=SPLIT_SEED)

[INFO]  Splitting dataset with seed 42...
[INFO]  Train Set: 76  | Tumor Ratio: 0.352
[INFO]  Valid Set: 17  | Tumor Ratio: 0.360
[INFO]  Test Set:  17  | Tumor Ratio: 0.372


In [9]:
train_dataloader, val_dataloader, test_dataloader = get_dataloaders(train_patients, val_patients, test_patients, batch_size=1)

In [11]:
print(f"Number of training samples:   {len(train_dataloader.dataset)}")
print(f"Number of validation samples: {len(val_dataloader.dataset)}")
print(f"Number of test samples:       {len(test_dataloader.dataset)}")

Number of training samples:   2784
Number of validation samples: 523
Number of test samples:       622


In [15]:
mri_img, label = train_dataloader.dataset[0]
print(label)
print(mri_img.shape)

0
torch.Size([3, 224, 224])


In [23]:
def readable_label(y_label):
    if y_label == 0:
        return "No Tumor"
    elif y_label == 1:
        return "Tumor"
    else:
        return "Unknown Label"

In [ ]:
image_indices = [7, 100, 225, 500]

for i in image_indices:
    # Get data
    train_img, train_label = train_dataloader.dataset[i]
    valid_img, valid_label = val_dataloader.dataset[i]
    test_img, test_label = test_dataloader.dataset[i]

    # Denormalize
    train_img = (train_img * torch.tensor([0.229, 0.224, 0.225]).view(3,1,1) + torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)).clip(0, 1)
    valid_img = (valid_img * torch.tensor([0.229, 0.224, 0.225]).view(3,1,1) + torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)).clip(0, 1)
    test_img =( test_img * torch.tensor([0.229, 0.224, 0.225]).view(3,1,1) + torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)).clip(0, 1)

    # Reshape from (C, H, W) -> (H, W, C)
    train_img_view = torch.permute(train_img, (1, 2, 0))
    valid_img_view = torch.permute(valid_img, (1, 2, 0))
    test_img_view = torch.permute(test_img, (1, 2, 0))
    # Display in matplotlib
    fig, axs = plt.subplots(1, 3)

    # axs[0] refers to the first (left) subplot
    axs[0].imshow(train_img_view)
    axs[0].set_title(f"Train Set -> {readable_label(train_label)}") # Optional: Add subplot title
    axs[0].axis('off') # Optional: Hide the axis ticks

    # axs[1] refers to the second (middle) subplot
    axs[1].imshow(valid_img_view)
    axs[1].set_title(f"Validation Set -> {readable_label(valid_label)}") # Optional: Add subplot title
    axs[1].axis('off') # Optional: Hide the axis ticks

    # axs[1] refers to the second (right) subplot
    axs[2].imshow(test_img_view)
    axs[2].set_title(f"Test Set -> {readable_label(test_label)}") # Optional: Add subplot title
    axs[2].axis('off') # Optional: Hide the axis ticks

    # 4. Adjust layout and show the figure
    plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent title overlap
    plt.show()